## mount drive

In [7]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

## define folder path

In [ ]:
import os

drive_path = "/content/drive/MyDrive/DeepLearning"

os.makedirs(drive_path, exist_ok=True)

print("Dataset will be saved in:", drive_path)

Dataset will be saved in: /content/drive/MyDrive/DeepLearning


## train dataset - coco 2014

In [ ]:
!wget "http://images.cocodataset.org/zips/train2014.zip" -O dataset.zip

--2026-06-08 15:43:31--  http://images.cocodataset.org/zips/train2014.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.253.89, 16.15.255.236, 52.216.37.225, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.253.89|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13510573713 (13G) [application/zip]
Saving to: ‘dataset.zip’

dataset.zip           0%[                    ]  88.56M  17.3MB/s    eta 17m 11s^C


In [ ]:
import zipfile

zip_file = "/content/dataset.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(drive_path)

print("Extraction completed.")

Extraction completed.


## Val dataset - coco 2014

In [ ]:
!wget "http://images.cocodataset.org/zips/val2014.zip" -O val.zip

--2026-06-04 13:19:11--  http://images.cocodataset.org/zips/val2014.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 52.216.49.1, 52.217.91.180, 52.216.243.52, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|52.216.49.1|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6645013297 (6.2G) [application/zip]
Saving to: ‘val.zip’

val.zip             100%[===================>]   6.19G  45.5MB/s    in 2m 18s  

2026-06-04 13:21:29 (46.0 MB/s) - ‘val.zip’ saved [6645013297/6645013297]



In [ ]:
import zipfile

zip_file = "/content/val.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(drive_path)

print("Extraction completed.")

Extraction completed.


## anottations - cooc 2014

In [ ]:
!wget "http://images.cocodataset.org/annotations/annotations_trainval2014.zip" -O anot.zip

--2026-06-04 14:37:50--  http://images.cocodataset.org/annotations/annotations_trainval2014.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.253.100, 16.15.254.67, 16.182.73.9, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.253.100|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252872794 (241M) [application/zip]
Saving to: ‘anot.zip’

anot.zip            100%[===================>] 241.16M  39.8MB/s    in 6.4s    

2026-06-04 14:37:56 (37.6 MB/s) - ‘anot.zip’ saved [252872794/252872794]



In [ ]:
import zipfile

zip_file = "/content/anot.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(drive_path)

print("Extraction completed.")

Extraction completed.


## test dataset - coco 2014

In [ ]:
!wget "http://images.cocodataset.org/zips/test2014.zip" -O test.zip

--2026-06-08 15:43:52--  http://images.cocodataset.org/zips/test2014.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.236.175, 54.231.203.233, 52.216.32.89, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.236.175|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6660437059 (6.2G) [application/zip]
Saving to: ‘test.zip’

test.zip              0%[                    ]  29.97M  9.05MB/s    eta 12m 12s^C


In [ ]:
import zipfile

zip_file = "/content/test.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(drive_path)

print("Extraction completed.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/test.zip'

## structure of folder

In [ ]:
for root, dirs, files in os.walk(drive_path):
    print(root)

/content/drive/MyDrive/DeepLearning
/content/drive/MyDrive/DeepLearning/train2014
/content/drive/MyDrive/DeepLearning/val2014
/content/drive/MyDrive/DeepLearning/annotations
/content/drive/MyDrive/DeepLearning/test2014


In [ ]:
# Install required packages
!pip install -q pycocoevalcap pycocotools nltk einops
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [ ]:
#making enviroment
import os, json, math, random, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

In [ ]:
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


#  COCO paths
DRIVE_PATH  = '/content/drive/MyDrive/DeepLearning'
TRAIN_DIR   = os.path.join(DRIVE_PATH, 'train2014')
VAL_DIR     = os.path.join(DRIVE_PATH, 'val2014')
ANNOT_DIR   = os.path.join(DRIVE_PATH, 'annotations')
TRAIN_ANNOT = os.path.join(ANNOT_DIR, 'captions_train2014.json')
VAL_ANNOT   = os.path.join(ANNOT_DIR, 'captions_val2014.json')

CKPT_DIR = os.path.join(DRIVE_PATH, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

print('📁 Paths:')
for name, path in [('Train images', TRAIN_DIR), ('Val images', VAL_DIR),
                    ('Train annotations', TRAIN_ANNOT), ('Val annotations', VAL_ANNOT)]:
    exists = '✅' if os.path.exists(path) else '❌'
    print(f'  {exists} {name}: {path}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Paths:
  ✅ Train images: /content/drive/MyDrive/DeepLearning/train2014
  ✅ Val images: /content/drive/MyDrive/DeepLearning/val2014
  ✅ Train annotations: /content/drive/MyDrive/DeepLearning/annotations/captions_train2014.json
  ✅ Val annotations: /content/drive/MyDrive/DeepLearning/annotations/captions_val2014.json


In [6]:
class Vocabulary:
    """Maps tokens ↔ integer indices. Special tokens: <PAD>=0, <SOS>=1, <EOS>=2, <UNK>=3."""

    PAD, SOS, EOS, UNK = 0, 1, 2, 3

    def __init__(self, freq_threshold: int = 5):
        self.freq_threshold = freq_threshold
        self.itos = {0:'<PAD>', 1:'<SOS>', 2:'<EOS>', 3:'<UNK>'}
        self.stoi = {v:k for k,v in self.itos.items()}

    def __len__(self): return len(self.itos)

    @staticmethod
    def tokenize(text: str) -> List[str]:
        return text.lower().strip().rstrip('.').split()

    def build(self, captions: List[str]):
        counter = Counter()
        for cap in captions: counter.update(self.tokenize(cap))
        idx = 4
        for word, freq in sorted(counter.items()):
            if freq >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx]  = word
                idx += 1
        print(f'📖 Vocabulary: {len(self)} tokens')

    def encode(self, text: str, max_len: int = 52) -> List[int]:
        tokens = [self.SOS] + \
                 [self.stoi.get(w, self.UNK) for w in self.tokenize(text)][:max_len-2] + \
                 [self.EOS]
        return tokens

    def decode(self, indices: List[int]) -> str:
        words = []
        for i in indices:
            if i == self.EOS: break
            if i not in (self.PAD, self.SOS): words.append(self.itos.get(i, '<UNK>'))
        return ' '.join(words)

# Build vocabulary from training annotations
with open(TRAIN_ANNOT) as f:
    train_data = json.load(f)

all_captions = [a['caption'] for a in train_data['annotations']]
vocab = Vocabulary(freq_threshold=5)
vocab.build(all_captions)

NameError: name 'List' is not defined

In [ ]:
IMG_SIZE    = 224
MAX_LEN     = 52
BATCH_SIZE  = 32
MAX_SAMPLES = 50_000   # set None to use full dataset (~414k)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class COCOCaptionDataset(Dataset):
    def __init__(self, img_dir, annot_path, vocab, transform, max_samples=None):
        with open(annot_path) as f:
            data = json.load(f)
        id2file = {img['id']: img['file_name'] for img in data['images']}
        annots  = data['annotations']
        if max_samples: annots = annots[:max_samples]
        self.samples = []
        for a in annots:
            fname = id2file.get(a['image_id'])
            fpath = os.path.join(img_dir, fname)
            if fname and os.path.exists(fpath):
                self.samples.append((fpath, a['caption']))
        self.vocab = vocab; self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fpath, caption = self.samples[idx]
        img = Image.open(fpath).convert('RGB')
        return self.transform(img), torch.tensor(self.vocab.encode(caption, MAX_LEN), dtype=torch.long)

def collate_fn(batch):
    imgs, caps = zip(*batch)
    imgs = torch.stack(imgs)
    max_len = max(c.size(0) for c in caps)
    padded = torch.zeros(len(caps), max_len, dtype=torch.long)
    for i, c in enumerate(caps): padded[i, :c.size(0)] = c
    return imgs, padded

train_ds = COCOCaptionDataset(TRAIN_DIR, TRAIN_ANNOT, vocab, train_transform, MAX_SAMPLES)
val_ds   = COCOCaptionDataset(VAL_DIR,   VAL_ANNOT,   vocab, val_transform,   MAX_SAMPLES//10)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, collate_fn=collate_fn)

In [ ]:
class EfficientNetEncoder(nn.Module):
    """
    EfficientNetV2-S visual encoder.
    Output: (B, num_patches, embed_dim)  — e.g. (B, 49, 256)
    """
    def __init__(self, embed_dim: int = 256, fine_tune: bool = False):
        super().__init__()
        weights  = EfficientNet_V2_S_Weights.IMAGENET1K_V1
        backbone = efficientnet_v2_s(weights=weights)

        self.features  = backbone.features  # → (B, 1280, 7, 7)
        self.proj = nn.Sequential(
            nn.Linear(1280, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )
        # Learnable spatial position embedding for 7×7 = 49 patches
        self.pos_embed = nn.Parameter(torch.randn(1, 49, embed_dim) * 0.02)
        self.set_fine_tune(fine_tune)

    def set_fine_tune(self, enable: bool):
        for p in self.features.parameters(): p.requires_grad = False
        if enable:
            for i in [6, 7]:  # Unfreeze last 2 MBConv blocks
                for p in self.features[i].parameters(): p.requires_grad = True

    def forward(self, x):
        feat = self.features(x)               # (B, 1280, 7, 7)
        B, C, H, W = feat.shape
        feat = feat.flatten(2).permute(0, 2, 1)  # (B, 49, 1280)
        feat = self.proj(feat)                 # (B, 49, embed_dim)
        return feat + self.pos_embed            # add spatial positions

# Quick sanity check
enc_test = EfficientNetEncoder(embed_dim=256).to(DEVICE)
x_test   = torch.randn(2, 3, 224, 224).to(DEVICE)
out_test = enc_test(x_test)
print(f'Encoder output: {out_test.shape}')  # → torch.Size([2, 49, 256])
del enc_test, x_test, out_test

Encoder output: torch.Size([2, 49, 256])


In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=8,
                 num_layers=6, ff_dim=1024, max_len=52, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos   = nn.Embedding(max_len, embed_dim)
        self.drop  = nn.Dropout(dropout)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True,  # Pre-LN for stability
            activation='gelu'
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)

        # Weight tying: embedding weights shared with output head
        self.head.weight = self.embed.weight

        nn.init.normal_(self.embed.weight, std=0.02)
        nn.init.normal_(self.pos.weight,   std=0.02)

    def forward(self, tgt_ids, memory, tgt_key_padding_mask=None):
        T  = tgt_ids.size(1)
        pos = torch.arange(T, device=tgt_ids.device).unsqueeze(0)
        x   = self.drop(self.embed(tgt_ids) + self.pos(pos))

        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        out = self.transformer(x, memory,
                               tgt_mask=causal_mask,
                               tgt_key_padding_mask=tgt_key_padding_mask)
        return self.head(self.norm(out))   # (B, T, vocab_size)

NameError: name 'nn' is not defined

In [ ]:
class ImageCaptioningModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=8,
                 num_layers=6, ff_dim=1024, max_len=52, dropout=0.1):
        super().__init__()
        self.encoder = EfficientNetEncoder(embed_dim=embed_dim)
        self.decoder = TransformerDecoder(vocab_size, embed_dim, num_heads,
                                            num_layers, ff_dim, max_len, dropout)

    def forward(self, images, captions):
        # Teacher forcing: input = cap[:-1], target = cap[1:]
        memory   = self.encoder(images)
        tgt      = captions[:, :-1]
        pad_mask = (tgt == 0)
        return self.decoder(tgt, memory, pad_mask)

    @torch.no_grad()
    def generate_greedy(self, image, max_len=30):
        self.eval()
        memory = self.encoder(image.unsqueeze(0).to(DEVICE))
        tokens = [vocab.SOS]
        for _ in range(max_len):
            tgt      = torch.tensor([tokens], device=DEVICE)
            next_id  = self.decoder(tgt, memory)[0, -1].argmax().item()
            if next_id == vocab.EOS: break
            tokens.append(next_id)
        return vocab.decode(tokens[1:])

    @torch.no_grad()
    def generate_beam(self, image, beam_size=5, max_len=30):
        self.eval()
        memory = self.encoder(image.unsqueeze(0).to(DEVICE))
        beams, completed = [(0.0, [vocab.SOS])], []
        for _ in range(max_len):
            candidates = []
            for score, seq in beams:
                if seq[-1] == vocab.EOS: completed.append((score, seq)); continue
                tgt      = torch.tensor([seq], device=DEVICE)
                log_prob = F.log_softmax(self.decoder(tgt, memory)[0, -1], dim=-1)
                for lp, idx in zip(*log_prob.topk(beam_size)):
                    candidates.append((score + lp.item(), seq + [idx.item()]))
            if not candidates: break
            candidates.sort(key=lambda x: x[0] / len(x[1]), reverse=True)
            beams = candidates[:beam_size]
        best = max(completed + beams, key=lambda x: x[0] / max(len(x[1]), 1))
        return vocab.decode(best[1][1:])

# Instantiate
model = ImageCaptioningModel(
    vocab_size=len(vocab), embed_dim=256, num_heads=8,
    num_layers=6, ff_dim=1024, max_len=MAX_LEN, dropout=0.1
).to(DEVICE)

In [ ]:
import math, time

def get_optimizer_and_scheduler(model, phase, steps_per_epoch):
    if phase == 1:
        lr, epochs = 3e-4, 8
    else:
        lr, epochs = 1e-4, 20

    params    = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    total_steps  = epochs * steps_per_epoch
    warmup_steps = steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler, epochs


criterion = nn.CrossEntropyLoss(ignore_index=vocab.PAD, label_smoothing=0.1)


def run_epoch(model, loader, optimizer, scheduler, is_train):
    model.train() if is_train else model.eval()
    total_loss, total_tok = 0.0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for imgs, caps in loader:
            imgs, caps = imgs.to(DEVICE), caps.to(DEVICE)
            logits  = model(imgs, caps)
            targets = caps[:, 1:]
            B, T, V = logits.shape
            loss    = criterion(logits.reshape(B*T, V), targets.reshape(B*T))

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            mask = (targets != vocab.PAD)
            total_loss += loss.item() * mask.sum().item()
            total_tok  += mask.sum().item()

    avg_loss = total_loss / total_tok
    return avg_loss, math.exp(min(avg_loss, 20))


def train_phase(model, phase, train_loader, val_loader):
    print(f'\n{"="*55}')
    print(f'  PHASE {phase} TRAINING')
    print(f'{"="*55}')

    optimizer, scheduler, n_epochs = get_optimizer_and_scheduler(
        model, phase, len(train_loader))

    best_val, history = float('inf'), []

    for epoch in range(1, n_epochs + 1):
        t0 = time.time()
        tr_loss, tr_ppl = run_epoch(model, train_loader, optimizer, scheduler, True)
        va_loss, va_ppl = run_epoch(model, val_loader,   optimizer, scheduler, False)
        elapsed = time.time() - t0

        history.append({'epoch': epoch, 'train_loss': tr_loss,
                        'val_loss': va_loss, 'train_ppl': tr_ppl, 'val_ppl': va_ppl})

        print(f'Ep {epoch:02d}/{n_epochs} | '
              f'Train Loss: {tr_loss:.4f} (PPL {tr_ppl:.1f}) | '
              f'Val Loss: {va_loss:.4f} (PPL {va_ppl:.1f}) | '
              f'{elapsed:.0f}s')

        if va_loss < best_val:
            best_val = va_loss
            ckpt = os.path.join(CKPT_DIR, f'best_phase{phase}.pt')
            torch.save({'epoch': epoch, 'model': model.state_dict(),
                        'val_loss': va_loss}, ckpt)
            print(f'   💾 Saved checkpoint (val_loss={va_loss:.4f})')

    return history


# ── Phase 1: Frozen encoder ──────────────────────────────────
history_p1 = train_phase(model, phase=1,
                         train_loader=train_loader,
                         val_loader=val_loader)

# ── Phase 2: Fine-tune encoder ───────────────────────────────
model.encoder.set_fine_tune(enable=True)
history_p2 = train_phase(model, phase=2,
                         train_loader=train_loader,
                         val_loader=val_loader)


  PHASE 1 TRAINING
Ep 01/8 | Train Loss: 5.3231 (PPL 205.0) | Val Loss: 4.3566 (PPL 78.0) | 2721s
   💾 Saved checkpoint (val_loss=4.3566)
Ep 02/8 | Train Loss: 4.0012 (PPL 54.7) | Val Loss: 3.9719 (PPL 53.1) | 831s
   💾 Saved checkpoint (val_loss=3.9719)
Ep 03/8 | Train Loss: 3.7347 (PPL 41.9) | Val Loss: 3.8448 (PPL 46.7) | 838s
   💾 Saved checkpoint (val_loss=3.8448)
Ep 04/8 | Train Loss: 3.5857 (PPL 36.1) | Val Loss: 3.7704 (PPL 43.4) | 830s
   💾 Saved checkpoint (val_loss=3.7704)
Ep 05/8 | Train Loss: 3.4769 (PPL 32.4) | Val Loss: 3.7323 (PPL 41.8) | 833s
   💾 Saved checkpoint (val_loss=3.7323)
Ep 06/8 | Train Loss: 3.3931 (PPL 29.8) | Val Loss: 3.7142 (PPL 41.0) | 856s
   💾 Saved checkpoint (val_loss=3.7142)
Ep 07/8 | Train Loss: 3.3346 (PPL 28.1) | Val Loss: 3.7104 (PPL 40.9) | 835s
   💾 Saved checkpoint (val_loss=3.7104)
Ep 08/8 | Train Loss: 3.3075 (PPL 27.3) | Val Loss: 3.7070 (PPL 40.7) | 823s
   💾 Saved checkpoint (val_loss=3.7070)

  PHASE 2 TRAINING
Ep 01/20 | Train Loss: